In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

In [2]:
if torch.cuda.is_available():
    # Print the name of the GPU
    print(f"GPU: {torch.cuda.get_device_name(0)} is available.")
    # Print the number of available CUDA devices
    print(f"Number of GPUs available: {torch.cuda.device_count()}")
else:
    print("No GPU available. Training will run on CPU.")


No GPU available. Training will run on CPU.


In [3]:
d_model = 4
key_dimension = 2

In [4]:
w_q = nn.Linear(d_model, key_dimension, bias=False)
w_k = nn.Linear(d_model, key_dimension, bias=False)
w_v = nn.Linear(d_model, key_dimension, bias=False)
w_o = nn.Linear(key_dimension, d_model, bias=False)

In [5]:
dummy_input = torch.rand(2, 3, 4)
print(dummy_input.shape)
print("(batch_size, num_tokens, embedding_dimension)")
dummy_input

torch.Size([2, 3, 4])
(batch_size, num_tokens, embedding_dimension)


tensor([[[0.3150, 0.1323, 0.9917, 0.9843],
         [0.7040, 0.1500, 0.4378, 0.1264],
         [0.6365, 0.6604, 0.9066, 0.7964]],

        [[0.3403, 0.3420, 0.4446, 0.8521],
         [0.6707, 0.9867, 0.7330, 0.4928],
         [0.5552, 0.1842, 0.9851, 0.1930]]])

In [6]:
q = w_q(dummy_input)
k = w_k(dummy_input)
v = w_v(dummy_input)

In [7]:
print(q.shape)
print("(batch_size, num_tokens, key_dimension)")
q

torch.Size([2, 3, 2])
(batch_size, num_tokens, key_dimension)


tensor([[[0.8634, 0.4063],
         [0.1976, 0.2053],
         [0.8821, 0.5098]],

        [[0.7485, 0.5260],
         [0.8593, 0.6361],
         [0.4639, 0.2503]]], grad_fn=<ViewBackward0>)

In [8]:
print(k.shape)
k

torch.Size([2, 3, 2])


tensor([[[0.3906, 0.6605],
         [0.2714, 0.3258],
         [0.5604, 0.6892]],

        [[0.2035, 0.6202],
         [0.5368, 0.7147],
         [0.4434, 0.4781]]], grad_fn=<ViewBackward0>)

In [9]:
k_t = k.transpose(-2, -1)
print(k_t.shape)
k_t

torch.Size([2, 2, 3])


tensor([[[0.3906, 0.2714, 0.5604],
         [0.6605, 0.3258, 0.6892]],

        [[0.2035, 0.5368, 0.4434],
         [0.6202, 0.7147, 0.4781]]], grad_fn=<TransposeBackward0>)

In [10]:
sim1 = (q @ k.transpose(-2, -1))/math.sqrt(key_dimension)
sim1
# Attention Scores Matrix

tensor([[[0.4282, 0.2593, 0.5401],
         [0.1504, 0.0852, 0.1783],
         [0.4818, 0.2867, 0.5980]],

        [[0.3384, 0.5499, 0.4125],
         [0.4026, 0.6476, 0.4844],
         [0.1765, 0.3026, 0.2301]]], grad_fn=<DivBackward0>)

In [11]:
sim2 = F.softmax(sim1, dim=-1)
sim2
# Attention Weights matrix
# Basically building a neural network for value vectors

tensor([[[0.3375, 0.2850, 0.3775],
         [0.3373, 0.3160, 0.3468],
         [0.3394, 0.2793, 0.3813]],

        [[0.3019, 0.3730, 0.3251],
         [0.2974, 0.3799, 0.3227],
         [0.3135, 0.3557, 0.3308]]], grad_fn=<SoftmaxBackward0>)

In [12]:
v # Value vectors

tensor([[[ 0.0071,  0.0156],
         [-0.6218, -0.0551],
         [-0.3125,  0.0224]],

        [[-0.2490,  0.1007],
         [-0.5401,  0.0487],
         [-0.3779, -0.0990]]], grad_fn=<ViewBackward0>)

In [13]:
sim3 = sim2 @ v
sim3 
# Applying Attention Weights to value vectors
# Essentially, passing value vectors through a neural network

tensor([[[-0.2928, -0.0020],
         [-0.3024, -0.0044],
         [-0.2904, -0.0015]],

        [[-0.3995,  0.0164],
         [-0.4012,  0.0165],
         [-0.3952,  0.0161]]], grad_fn=<UnsafeViewBackward0>)

In [14]:
output = w_o(sim3)
# Projecting from key_dimension back to d_model (up projection)
print(output.shape)
output
# We have output tensor of the same dimension as our input. Ready to be passed on to the next layer.

torch.Size([2, 3, 4])


tensor([[[0.3430, 0.4156, 0.6395, 0.1362],
         [0.3492, 0.4129, 0.6383, 0.1340],
         [0.3415, 0.4161, 0.6397, 0.1367]],

        [[0.4162, 0.4169, 0.6368, 0.1172],
         [0.4173, 0.4168, 0.6367, 0.1169],
         [0.4133, 0.4172, 0.6370, 0.1180]]], grad_fn=<ViewBackward0>)

In [17]:
class SelfAttention(nn.Module):
    """
    Compute Scaled Dot Product Attention
    """

    def __init__(self, d_model, key_dimension):
        super().__init__()
        self.d_model = d_model
        self.key_dimension = key_dimension
        self.w_q = nn.Linear(d_model, key_dimension, bias=False)
        self.w_k = nn.Linear(d_model, key_dimension, bias=False)
        self.w_v = nn.Linear(d_model, key_dimension, bias=False)
        self.w_o = nn.Linear(key_dimension, d_model, bias=False)

    def forward(self, input: torch.Tensor) -> torch.Tensor:
        q = self.w_q(input)
        k = self.w_k(input)
        v = self.w_v(input)
        

        sim1 = (q @ k.transpose(-2, -1))/math.sqrt(key_dimension)
        sim2 = F.softmax(sim1, dim=1)
        sim3 = sim2 @ v
        output = w_o(sim3)
        return output

In [18]:
dummy_input = torch.rand(1, 3, 4)
dummy_input

tensor([[[0.5601, 0.2019, 0.5892, 0.7882],
         [0.3330, 0.0206, 0.1108, 0.3379],
         [0.9551, 0.9398, 0.6194, 0.5328]]])

In [19]:
attention = SelfAttention(d_model, key_dimension)
attention

SelfAttention(
  (w_q): Linear(in_features=4, out_features=2, bias=True)
  (w_k): Linear(in_features=4, out_features=2, bias=True)
  (w_v): Linear(in_features=4, out_features=2, bias=True)
  (w_o): Linear(in_features=2, out_features=4, bias=True)
)

In [20]:
dummy_output = attention(dummy_input)
dummy_output

tensor([[[-0.1953, -0.2857,  0.4276,  0.1411],
         [-0.1727, -0.2348,  0.4436,  0.1451],
         [-0.2054, -0.3088,  0.4202,  0.1393]]], grad_fn=<ViewBackward0>)